In [1]:
%pip install -U langchain-groq langchain-core langchain-community requests pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.4 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pydantic]1/2 [pydantic]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
import os


In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [5]:
print(multiply.invoke({'a':5, 'b':4}))

20


In [6]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## tool binding

In [7]:
llm = ChatGroq(model="openai/gpt-oss-120b")


In [10]:
llm.invoke('hi')

AIMessage(content='Hello! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "hi". We should greet back and ask how can help. Keep friendly.'}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 72, 'total_tokens': 120, 'completion_time': 0.099435961, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.002872141, 'prompt_tokens_details': None, 'queue_time': 0.344134204, 'total_time': 0.102308102}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12a98-2b88-70e2-b7ba-41dc2d58fde2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 48, 'total_tokens': 120, 'output_token_details': {'reasoning': 28}})

In [9]:
llm_with_tools = llm.bind_tools([multiply])

In [15]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hey there! I'm doing great—thanks for asking. How can I help you today?", additional_kwargs={'reasoning_content': 'User just greets. Should respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 133, 'total_tokens': 170, 'completion_time': 0.077123553, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.005797499, 'prompt_tokens_details': None, 'queue_time': 0.332339516, 'total_time': 0.082921052}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_60e4b492db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12aa6-fd1c-7733-b1b1-0833dc2029a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 37, 'total_tokens': 170, 'output_token_details': {'reasoning': 10}})

In [43]:
query = HumanMessage('can you multiply 7 with 10')

In [44]:
messages = [query]

In [45]:
messages

[HumanMessage(content='can you multiply 7 with 10', additional_kwargs={}, response_metadata={})]

In [46]:
result = llm_with_tools.invoke(messages)

In [47]:
messages.append(result)

In [48]:
messages

[HumanMessage(content='can you multiply 7 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to multiply 7 and 10 using the provided function. Use the multiply tool.', 'tool_calls': [{'id': 'fc_8a8f7480-427c-4df4-8903-f1662329727d', 'function': {'arguments': '{"a":7,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 137, 'total_tokens': 190, 'completion_time': 0.110013631, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005140073, 'prompt_tokens_details': None, 'queue_time': 0.384861684, 'total_time': 0.115153704}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12ab3-2af8-7661-b260-c33e3ebb3290-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 10}, 'id':

In [49]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 7, 'b': 10},
 'id': 'fc_8a8f7480-427c-4df4-8903-f1662329727d',
 'type': 'tool_call'}

In [50]:
tool_result = multiply.invoke(result.tool_calls[0])

In [51]:
tool_result

ToolMessage(content='70', name='multiply', tool_call_id='fc_8a8f7480-427c-4df4-8903-f1662329727d')

In [52]:
messages.append(tool_result)

In [53]:
messages

[HumanMessage(content='can you multiply 7 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to multiply 7 and 10 using the provided function. Use the multiply tool.', 'tool_calls': [{'id': 'fc_8a8f7480-427c-4df4-8903-f1662329727d', 'function': {'arguments': '{"a":7,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 137, 'total_tokens': 190, 'completion_time': 0.110013631, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005140073, 'prompt_tokens_details': None, 'queue_time': 0.384861684, 'total_time': 0.115153704}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12ab3-2af8-7661-b260-c33e3ebb3290-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 10}, 'id':

In [54]:
llm_with_tools.invoke(messages).content

'The product of 7 and 10 is **70**.'